In [ ]:
from pathlib import Path
import geopandas as gpd
from shapely.geometry import LineString, Polygon, box
import rasterio
import os
import matplotlib.pyplot as plt
import folium
from branca.colormap import LinearColormap
import os
import numpy as np
from shapely.ops import transform
import pyproj

In [ ]:
from post_processing_functions import load_traffic_centers, process_all_regions, save_results

def run_flood_analysis_multi_buffer():
    # Configuration
    traffic_centers_file = r"P:\bovenregionale-stresstest-hwn\Data\Traffic_centrals\Traffic_centers.xlsx"
    #hazard_maps_base_path = r"P:\bovenregionale-stresstest-hwn\Data\Hazard_maps\Hazard_maps-in_use" #change this to add regions from analysis
    hazard_maps_base_path = r"P:\bovenregionale-stresstest-hwn\Analysis"
    output_directory = r"P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Central_Analysis"
    region_list = ["ARK-NZK", "Vallei en Veluwe", "Noord-Westelijke Delta", "Noord-Brabant Oost"]
    buffer_distances = [5, 10, 50, 100, 200]  # meters

    all_summaries = {}

    for buffer_distance in buffer_distances:
        print(f"\n=== Running analysis for buffer distance: {buffer_distance}m ===")
        # Step 1: Load and buffer traffic centers
        gdf_buffered = load_traffic_centers(traffic_centers_file, buffer_distance)

        # Step 2: Process all regions and flood maps
        gdf_with_floods, all_results = process_all_regions(gdf_buffered, region_list, hazard_maps_base_path)

        # Step 3: Save results in a subfolder for each buffer distance
        buffer_output_dir = f"{output_directory}\\buffer_{buffer_distance}m"
        gpkg_path, excel_path = save_results(gdf_with_floods, buffer_output_dir)

        # Generate summary statistics
        summary_stats = {
            'total_traffic_centers': len(gdf_with_floods),
            'centers_with_flood_data': (gdf_with_floods['max_flood_depth'] != -9999).sum(),
            'centers_with_flooding': (gdf_with_floods['max_flood_depth'] > 0).sum(),
            'max_flood_depth_found': gdf_with_floods['max_flood_depth'].max(),
            'mean_flood_depth': gdf_with_floods[gdf_with_floods['max_flood_depth'] != -9999]['max_flood_depth'].mean(),
            'gpkg_path': gpkg_path,
            'excel_path': excel_path
        }
        all_summaries[buffer_distance] = summary_stats

        print(f"\nAnalysis complete for buffer {buffer_distance}m!")
        print(f"Processed {len(all_results)} flood maps across {len(region_list)} regions.")

    return all_summaries

summaries = run_flood_analysis_multi_buffer()

In [ ]:
region_list = ["Noord-Westelijke Delta","Vallei en Veluwe","Achterhoek", "Brabantse Delta","Friesland","Groningen en NO-Drenthe","Limburg","ARK-NZK",  "Noord-Brabant Oost",
               "Rivierenland","Scheldestromen"
               ]

In [ ]:
region_list = ["Noord-Westelijke Delta"]

In [8]:
#Linking traffic data road segments
import pandas as pd
root_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Analysis_Data")
data = Path(r"P:\bovenregionale-stresstest-hwn\Data\Traffic_data\INWEVA_2024 (1)")
traffic_geometry_path=data.joinpath("INWEVA_2024_netwerk\INWEVA_2024_netwerk.shp") 
werkdag_traffic_csv_path = data.joinpath("inweva_werkdag_2024DEC.csv")
traffic_geometry= gpd.read_file(traffic_geometry_path, driver='SHP') 
traffic= pd.read_csv(werkdag_traffic_csv_path, sep=';') 

traffic_subset = traffic[['VBN_ID', 'AL_D_WR', 'VRPC_D_WR']]
traffic_subset['VBN_ID'] = traffic_subset['VBN_ID'].astype(str)
result = traffic_geometry.merge(traffic_subset, left_on='vbn_id', right_on='VBN_ID', how='left')
print(f"Original traffic_geometry rows: {len(traffic_geometry)}")
print(f"Result rows: {len(result)}")
print(result.columns.tolist())

# Save the result as a geopackage
# Rename the VBN_ID column to avoid conflicts
result_renamed = result.rename(columns={'VBN_ID': 'traffic_vbn_id'})

output_path = root_dir.joinpath("traffic_werkdag_2024DEC.gpkg")
result_renamed.to_file(output_path, driver='GPKG')
#print(f"Saved merged data to: {output_path}")

result_renamed = result.drop(columns=['VBN_ID'])# Rename the VBN_ID column to avoid conflicts
result_exploded = result_renamed.copy()# Split nwb_ids and create duplicate rows for each value
result_exploded['nwb_ids'] = result_exploded['nwb_ids'].str.split(', ')# Split the nwb_ids column by comma and expand into separate rows
result_exploded = result_exploded.explode('nwb_ids')
result_exploded = result_exploded.reset_index(drop=True)# Reset index after exploding
result_exploded['individual_nwb_id'] = result_exploded['nwb_ids']# Create a new column with individual nwb_id values
#print(f"Original rows: {len(result_renamed)}")
#print(f"Exploded rows: {len(result_exploded)}")

### check here!!!
# Select only the required columns from traffic and create a proper copy
#AL_D_WR = Totale INWEVA intensiteit over periode 'dag' (7:00 - 19:00 uur) alle voertuigen werkdag (in mvt)
#VRPC_D_WR = Vrachtpercentage dag werkdag, Freight percentage day working day
traffic_subset_ver01 = result_exploded[['individual_nwb_id', 'AL_D_WR', 'VRPC_D_WR']].copy() 
# Convert both columns to string to ensure they match
traffic_subset_ver01['individual_nwb_id'] = traffic_subset_ver01['individual_nwb_id'].astype(str)

for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    road_path = root_dir / "damages/HZ_damage_segmented.gpkg"
    road = gpd.read_file(road_path, driver='GPKG') 
    #print(traffic.columns)
    #print(traffic_geometry.columns)
    print(road.columns)
    print(result_exploded.columns)

    # Also convert the WVK_ID column in road to string for consistent merge
    road_copy = road.copy()
    road_copy['WVK_ID'] = road_copy['WVK_ID'].astype(str)

    # Perform left join to keep all rows from road
    Road_merge = road_copy.merge(traffic_subset_ver01, left_on='WVK_ID', right_on='individual_nwb_id', how='left')

    # Display the result
    #print(f"Original road rows: {len(road)}")
    #print(f"Result rows: {len(Road_merge)}")
    #print(f"Road_merge columns: {Road_merge.columns.tolist()}")

    # Convert WVK_ID to string and remove decimal points
    road_copy['WVK_ID'] = road_copy['WVK_ID'].astype(str).str.replace('.0', '', regex=False)

    # Perform left join to keep all rows from road
    Road_merge = road_copy.merge(traffic_subset_ver01, left_on='WVK_ID', right_on='individual_nwb_id', how='left')

    # Display the result
    #print(f"Original road rows: {len(road)}")
    #print(f"Result rows: {len(Road_merge)}")
    #print(f"Road_merge columns: {Road_merge.columns.tolist()}")

    # Check data types and sample values after cleaning
    #print("\nAfter cleaning:")
    #print(f"Sample WVK_ID values from road: {road_copy['WVK_ID'].head().tolist()}")
    #print(f"Sample individual_nwb_id values from traffic: {traffic_subset_ver01['individual_nwb_id'].head().tolist()}")

    # Check for matches
    matched_records = Road_merge['individual_nwb_id'].notna().sum()
    total_records = len(Road_merge)
    match_percentage = (matched_records / total_records) * 100

    print(f"\nMatching results:")
    print(f"Total road records: {total_records}")
    print(f"Records with traffic data: {matched_records}")
    print(f"Records without traffic data: {total_records - matched_records}")
    print(f"Match percentage: {match_percentage:.2f}%")

    road_merge_output_path = root_dir.joinpath("hazard_overlay_with_traffic.gpkg")
    Road_merge.to_file(road_merge_output_path, driver='GPKG')
    print(f"Saved merged road and traffic data to: {road_merge_output_path}")



C:\Users\gunaratn\AppData\Local\Temp\ipykernel_9312\3853189403.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  traffic_subset['VBN_ID'] = traffic_subset['VBN_ID'].astype(str)


Original traffic_geometry rows: 9049
Result rows: 9049
['vbn_id', 'vbn_oms_tx', 'vbn_oms_bp', 'vbn_lengte', 'regio_cntr', 'regio_alt', 'nwb_ids', 'wegnrhmp_b', 'wegnrhmp_e', 'bpszijde_b', 'bpszijde_e', 'hectoltr_b', 'hectoltr_e', 'bnsubsrt_b', 'bnsubsrt_e', 'km_b', 'km_e', 'hm_midden', 'vbn_id_tgn', 'vias_baan', 'geometry', 'VBN_ID', 'AL_D_WR', 'VRPC_D_WR']


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('traffic_werkdag_2024DEC')) failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('traffic_werkdag_2024DEC')) failed: disk I/O error"


Processing region: Limburg network
Index(['u', 'v', 'key', 'link_id', 'REF_ID', 'infra_type', 'avgspeed', 'lanes',
       'id_NWB', 'BST_CODE_N', 'NWSNAAM_HW', 'NETWERKSCH', 'node_A', 'node_B',
       'edge_fid', 'rfid_c', 'rfid', 'length', 'time', 'F_EV1_mi', 'F_EV1_ma',
       'F_EV1_me', 'F_EV1_fr', 'road_type', 'lanes_copy', 'dam_EV1_HZ',
       'geometry'],
      dtype='object')
Index(['vbn_id', 'vbn_oms_tx', 'vbn_oms_bp', 'vbn_lengte', 'regio_cntr',
       'regio_alt', 'nwb_ids', 'wegnrhmp_b', 'wegnrhmp_e', 'bpszijde_b',
       'bpszijde_e', 'hectoltr_b', 'hectoltr_e', 'bnsubsrt_b', 'bnsubsrt_e',
       'km_b', 'km_e', 'hm_midden', 'vbn_id_tgn', 'vias_baan', 'geometry',
       'AL_D_WR', 'VRPC_D_WR', 'individual_nwb_id'],
      dtype='object')


KeyError: 'WVK_ID'

In [ ]:
#On off ramp analysis
from pathlib import Path
import geopandas as gpd
from post_processing_functions import cluster_connected,aggregate_clusters_to_points
#region_list = ["Friesland", "Vallei en Veluwe","Noord-Westelijke Delta","Limburg"]


for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    network_file = root_dir / "hazard_overlay_with_traffic.gpkg"
    network_gdf = gpd.read_file(network_file)
    ramps_gdf = network_gdf[network_gdf["BST_CODE_N"].isin(["AFR", "OPR"])]
    afr_gdf = ramps_gdf[ramps_gdf["BST_CODE_N"] == "AFR"].copy()
    opr_gdf = ramps_gdf[ramps_gdf["BST_CODE_N"] == "OPR"].copy()
    
    afr_gdf_clustered = cluster_connected(afr_gdf)
    output_gpkg = root_dir / "afr_LineSegments.gpkg"
    afr_gdf_clustered.to_file(output_gpkg, driver="GPKG")

    
    afr_gdf_aggregated = aggregate_clusters_to_points(afr_gdf_clustered, "F_EV1_ma", method="mean")
    output_gpkg_aggregated = root_dir / "afr_Points.gpkg"
    afr_gdf_aggregated.to_file(output_gpkg_aggregated, driver="GPKG")

    opr_gdf_clustered = cluster_connected(opr_gdf)
    output_gpkg = root_dir / "opr_LineSegments.gpkg"
    opr_gdf_clustered.to_file(output_gpkg, driver="GPKG")

    opr_gdf_aggregated = aggregate_clusters_to_points(opr_gdf_clustered, "F_EV1_ma", method="mean")
    output_gpkg_aggregated = root_dir / "opr_Points.gpkg"
    opr_gdf_aggregated.to_file(output_gpkg_aggregated, driver="GPKG")


In [ ]:
region_list = ["Limburg","Vallei en Veluwe","Achterhoek", "Brabantse Delta","Friesland","Limburg","ARK-NZK",  "Noord-Brabant Oost",
               "Rivierenland","Scheldestromen"
               ]

In [5]:
region_list = ["Limburg"]

In [6]:
from post_processing_functions import Thresholding_for_artefacts,Thresholding_for_artefacts,Filter_and_aggregate_flooded_segments_exposure, Filter_and_aggregate_flooded_segments_damage, calculate_overlay_percentages,get_z_height_optimized
import pandas as pd 
# add tunnels and bridge % columns to exposure and damage files and filter

data_dir = Path(r"P:\bovenregionale-stresstest-hwn\Data\Processed_data")


tunnels = data_dir.joinpath("filtered_tunnels.gpkg")
bridges = data_dir.joinpath("filtered_bridges.gpkg")
kunstinweg = data_dir.joinpath("kunstinweg.shp")
road_height_path = data_dir.joinpath("road_height_points.gpkg")

kunstinweg_gdf = gpd.read_file(kunstinweg)
kunstinweg_gdf['geometry'] = kunstinweg_gdf['geometry'].buffer(0.2) # to make sure lines are valid
kunstinweg_gdf.rename(columns={'OMSCHR': 'objecttekst'}, inplace=True)

kunstinweg_bridge = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'brug']
kunstinweg_tunnel = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'tunnel']


tunnels_gdf_kunstoverweg = gpd.read_file(tunnels)
bridges_gdf_kunstoverweg = gpd.read_file(bridges) 

bridges_gdf = gpd.GeoDataFrame(
    pd.concat([bridges_gdf_kunstoverweg, kunstinweg_bridge], ignore_index=True),
    crs=bridges_gdf_kunstoverweg.crs
)

tunnels_gdf = gpd.GeoDataFrame(
    pd.concat([tunnels_gdf_kunstoverweg, kunstinweg_tunnel], ignore_index=True),
    crs=tunnels_gdf_kunstoverweg.crs
)



# Define allowed values for bridges and tunnels (lowercased for case-insensitive matching)
allowed_bridges = [
    'aanbrug', 'brug', 'brug (beweegbaar)', 'brug (landbouw)', 'brug (vast)',
    'brug beton', 'brug beton in', 'brug beton over', 'brug beweegbaar',
    'brug hout in', 'brug in', 'brug in de toerit va', 'brug staal in',
    'brug vast', 'vaste brug'
]

allowed_tunnels = [
    'cervedict tunnel', 'open tunnelbak', 'tunnel', 'tunnel vlak',
    'tunnelbak', 'tunnelbak den kaat'
]

filtered_viaducts = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'viaduct']

# Convert to lowercase for case-insensitive comparison
allowed_bridges = [x.lower() for x in allowed_bridges]
allowed_tunnels = [x.lower() for x in allowed_tunnels]

# Filter bridges
filtered_gdf_brug = bridges_gdf[
    bridges_gdf['objecttekst'].str.lower().isin(allowed_bridges)
]

# Filter tunnels
filtered_gdf_tunnel_and_bridges = tunnels_gdf[
    tunnels_gdf['objecttekst'].str.lower().isin(allowed_tunnels + allowed_bridges)
]




In [7]:
from post_processing_functions import Thresholding_for_artefacts,Filter_and_aggregate_flooded_segments_exposure, Filter_and_aggregate_flooded_segments_damage, calculate_overlay_percentages,get_z_height_optimized


for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "hazard_overlay_with_traffic.gpkg"
    roads_ex_gdf = gpd.read_file(roads_ex)
    #roads_dm = root_dir / "damages\HZ_damage_segmented.gpkg"
    #roads_dm_gdf = gpd.read_file(roads_dm)


    points_gdf = gpd.read_file(road_height_path)
    points_gdf = points_gdf.to_crs(roads_ex_gdf.crs)
    roads_ex_gdf['Z_height'] = get_z_height_optimized(roads_ex_gdf, points_gdf, threshold=50.0)
    #roads_dm_gdf['Z_height'] = get_z_height_optimized(roads_dm_gdf, points_gdf, threshold=50.0)

    print("Calculating tunnel and bridge percentages...")
    Roads_assets = calculate_overlay_percentages(roads_ex_gdf, filtered_gdf_brug, filtered_gdf_tunnel_and_bridges,filtered_viaducts)
    #Roads_damage = calculate_overlay_percentages(roads_dm_gdf, filtered_gdf_brug, filtered_gdf_tunnel_and_bridges,filtered_viaducts)

    #Roads_exposure.to_file(root_dir / "base_network_hazard_overlay.gpkg", driver='GPKG')
    #Roads_damage.to_file(root_dir / "damages\HZ_damage_segmented_overlay.gpkg", driver='GPKG')
    print("Applying thresholding to remove artefacts...")
    #Thresholding_for_artefacts("","Failure", Roads_exposure, root_dir)
    Thresholding_for_artefacts("F_","Damages", Roads_assets, root_dir)
    
    print(Roads_assets.columns)
    print("Filtering and aggregating flooded segments...")
    Filter_and_aggregate_flooded_segments_exposure(Roads_assets, root_dir,"Failure", dissolve_col='NETWERKSCH')
    Filter_and_aggregate_flooded_segments_damage(Roads_assets, root_dir,"Damages", dissolve_col='NETWERKSCH')



Processing region: Limburg network


DriverError: P:\bovenregionale-stresstest-hwn\Analysis\Limburg\Outputs\hazard_overlay_with_traffic.gpkg: No such file or directory

In [ ]:
# getting high risk schakels
#-> for Failure and Damages

for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    AG_roads_ex = root_dir / "Failure_Aggregated.gpkg"
    AG_roads_ex_gdf = gpd.read_file(AG_roads_ex)
    AG_roads_dm = root_dir / "Damages_Aggregated.gpkg"
    AG_roads_dm_gdf = gpd.read_file(AG_roads_dm)
    
    #Rsk for failure
    #which ones has the highest fraction flooded
    AG_roads_ex_gdf['fr_flooded_risk_rank'] = AG_roads_ex_gdf['fraction_flooded'].rank(ascending=False, method='min')
    #Which ones has the highest max flood depth
    AG_roads_ex_gdf['Max_flooded_risk_rank'] = AG_roads_ex_gdf['EV1_me_max'].rank(ascending=False, method='min')
    #Which ones has the highest max flood depth
    AG_roads_ex_gdf['fracton_depth'] = AG_roads_ex_gdf['EV1_me_mean'] * AG_roads_ex_gdf['fraction_flooded']
    AG_roads_ex_gdf['fracton_depth_risk_rank'] = AG_roads_ex_gdf['fracton_depth'].rank(ascending=False, method='min')
    
    
    #Risk for damages
    AG_roads_dm_gdf['Lower_dm_risk_rank'] = AG_roads_dm_gdf['lower_dam_EV1_HZ_sum'].rank(ascending=False, method='min')
    AG_roads_dm_gdf['Upper_dm_risk_rank'] = AG_roads_dm_gdf['upper_dam_EV1_HZ_sum'].rank(ascending=False, method='min')
    
    AG_roads_ex_gdf.to_file(root_dir / "Failure_Aggregated_with_risk.gpkg", driver='GPKG')
    AG_roads_dm_gdf.to_file(root_dir / "Damages_Aggregated_with_risk.gpkg", driver='GPKG')

    